# BHCast Pipeline

This notebook implements the BHCast pipeline in three parts:
1. Frame Autoregressive Generation
2. Feature Computation
3. Parameter Inference

You can run this tutorial **entirely on CPU**, although using a GPU will make the autoregressive forecasting faster. The code below will automatically select `cuda` when available and fall back to `cpu` otherwise.

First, let's set up the configuration parameters that would typically be passed as command line arguments.

In [ ]:
import os
import tempfile
import shutil
import numpy as np
import joblib
import matplotlib.pyplot as plt
import torch

# Import all necessary functions and classes from the pipeline
from bhcast_pipeline import (
    load_image_file, 
    GRMHDInference, 
    write_forecast_to_hdf5, 
    extract_features, 
    load_label_metadata, 
    feature_vector_from_metrics,
    decode_prediction,
    DEFAULT_FEATURE_ORDER
)

# Automatically choose device: use CUDA if available, else CPU.
# You can also override this manually below if you prefer.
auto_device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Configuration Parameters (Replace with your actual paths/values)
config = {
    'input_image': 'path/to/your/image.npy',
    'checkpoint_path': 'path/to/unet/checkpoint.pt',
    'data_processor_path': 'path/to/data_processor_sgra.pth',
    'dataset_name': 'sgra',
    'device': auto_device,  # set to 'cpu' explicitly if you want to force CPU
    'in_channels': 1,
    'out_channels': 1,
    'forecast_horizon': 60,
    'spin_hint': 0.0,
    'inclination_hint': 0,
    'fov_uas': 200.0,
    'fov_m_sgra': 39.764,
    'fov_m_m87': 52.355,
    'dt': 5.0,
    'nftot_spiral': None,
    'r_interest_m': 5.2,
    'n_angles': 360,
    'ps_radius_min': 0.75,
    'ps_radius_max': 1.5,
    'ps_radius_num': 5,
    'calibrate_ring': False,
    'grad_threshold': 0.0,
    'smooth_sigma': 1.0,
    'xgb_model': 'path/to/xgboost_model.joblib',
    'label_map': 'path/to/label_map.pkl',
    'temp_dir': None,
    'keep_temp': False
}

## 1. Frame Autoregressive Generation
In this step, we load the initial frame and use the GRMHD Unet model to autoregressively forecast future frames.

In [ ]:
# Load input image
print(f"Loading image from {config['input_image']}...")
try:
    image = load_image_file(config['input_image'])
    if image.ndim == 3:
        image = image.squeeze(0)
    if image.ndim != 2:
        raise ValueError(f"Expected 2D image (H, W), got shape {image.shape}")
        
    plt.figure(figsize=(5, 5))
    plt.imshow(image, cmap='inferno')
    plt.title("Input Frame")
    plt.axis('off')
    plt.show()
    
    print("Initializing GRMHD inference model...")
    inference = GRMHDInference(
        checkpoint_path=config['checkpoint_path'],
        data_processor_path=config['data_processor_path'],
        in_channels=config['in_channels'],
        out_channels=config['out_channels'],
        device=config['device'],
    )

    print(f"Generating {config['forecast_horizon']} frame forecast...")
    forecast = inference.predict(frame=image, horizon=config['forecast_horizon'])

    if forecast.ndim == 4:
        if forecast.shape[1] > 1:
            forecast = forecast[:, 0]
        else:
            forecast = np.squeeze(forecast, axis=1)
    if forecast.ndim != 3:
        raise ValueError(f"Forecast must have shape (T, H, W); got {forecast.shape}")

    print(f"Forecast generated with shape: {forecast.shape}")
    
    fig, axes = plt.subplots(1, min(5, len(forecast)), figsize=(15, 3))
    if len(forecast) < 5:
        axes = [axes]
    for i, ax in enumerate(axes):
        idx = i * (len(forecast) // len(axes))
        ax.imshow(forecast[idx], cmap='inferno')
        ax.set_title(f"Forecast Step: {idx+1}")
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error in generation step: {e}")


## 2. Feature Computation
Save the generated forecast to temporary HDF5 files and extract plasma features (pattern speed, pitch angle, asymmetry, etc.).

In [ ]:
try:
    temp_root = config['temp_dir'] or tempfile.mkdtemp(prefix="bhcast_pipeline_")
    h5_dir = os.path.join(temp_root, "forecast_h5")
    fov_m = config['fov_m_sgra'] if config['dataset_name'] == 'sgra' else config['fov_m_m87']

    print(f"Writing forecast to temp HDF5 directory: {h5_dir}")
    write_forecast_to_hdf5(
        forecast=forecast,
        output_dir=h5_dir,
        spin_meta=config['spin_hint'],
        inclination_meta=config['inclination_hint'],
        fov_uas=config['fov_uas'],
        fov_m=fov_m,
        dt=config['dt'],
    )

    radius_factors = list(np.linspace(float(config['ps_radius_min']), float(config['ps_radius_max']), int(config['ps_radius_num'])))

    print("\nExtracting features from the forecast segment...")
    metrics = extract_features(
        h5_dir=h5_dir,
        forecast=forecast,
        spin_meta=config['spin_hint'],
        inclination_meta=config['inclination_hint'],
        fov_m=fov_m,
        nftot_spiral=config['nftot_spiral'],
        r_interest_m=config['r_interest_m'],
        n_angles=config['n_angles'],
        radius_factors=radius_factors,
        calibrate_ring=config['calibrate_ring'],
        grad_threshold=config['grad_threshold'],
        smooth_sigma=config['smooth_sigma'],
    )

    print("\nExtracted Plasma Features:")
    for key, value in metrics.items():
        if key in DEFAULT_FEATURE_ORDER:
            print(f"  {key}: {value}")
    
    if 'cylinder' in metrics and metrics['cylinder'] is not None:
        print("\nVisualizing Cylinder Plot for Pattern Speed...")
        cylinder = metrics['cylinder']
        plt.figure(figsize=(10, 6))
        plt.imshow(cylinder, aspect='auto', origin='lower', cmap='afmhot')
        plt.colorbar(label='Intensity')
        plt.title('Smoothed Normalized Cylinder Plot')
        plt.xlabel('Angle Index')
        plt.ylabel('Time / Frame Index')
        plt.show()
            
except Exception as e:
    print(f"Error in feature computation step: {e}")


## 3. Parameter Inference
Run the extracted features through a trained XGBoost classifier to predict the joint spin and inclination labels.

In [ ]:
try:
    print("Loading label metadata and generating feature vector...")
    index_to_pair, required_feature_order, class_to_spin = load_label_metadata(config['label_map'])
    feature_vec = feature_vector_from_metrics(metrics, required_feature_order)

    print(f"Loading XGBoost model from {config['xgb_model']}...")
    model = joblib.load(config['xgb_model'])
    probs = model.predict_proba(feature_vec.reshape(1, -1))[0]
    pred_idx = int(np.argmax(probs))
    confidence = float(np.max(probs))
    pred_spin, pred_inc = decode_prediction(pred_idx, index_to_pair, class_to_spin, config['dataset_name'])

    spin_class = "prograde (a* > 0)" if pred_spin > 0 else "retrograde (a* < 0)"
    if pred_inc in [10, 30]:
        inc_class = "positive face-on (10/30)"
    elif pred_inc in [50, 70]:
        inc_class = "positive edge-on (50/70)"
    elif pred_inc in [110, 130]:
        inc_class = "negative edge-on (110/130)"
    elif pred_inc in [150, 170]:
        inc_class = "negative face-on (150/170)"
    else:
        inc_class = f"unknown ({pred_inc})"

    print("\n=== BHCast Pipeline Inference Results ===")
    print(f"  Joint class index: {pred_idx}")
    print(f"  Confidence: {confidence:.3f}")
    print(f"  Spin: {spin_class}")
    print(f"  Inclination: {inc_class}")

    # Plotting probabilities
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(probs)), probs)
    plt.xlabel("Class Index")
    plt.ylabel("Probability")
    plt.title("XGBoost Prediction Probabilities")
    plt.show()

except Exception as e:
    print(f"Error in inference step: {e}")
    
finally:
    if 'config' in locals() and not config.get('keep_temp', False) and config.get('temp_dir') is None:
        if 'temp_root' in locals():
            print(f"\nCleaning up temporary directory {temp_root}...")
            shutil.rmtree(temp_root, ignore_errors=True)
    elif 'temp_root' in locals():
        print(f"\nTemporary artifacts retained at: {temp_root}")
